# Notebook 03: Transformer ブロック

Notebook 02 の Self-Attention に以下を追加して、完全な Transformer ブロックを組み立てます：

```
X
├─→ MultiHeadAttention(X) ─→ + X ─→ LayerNorm ─→ X'
│                                                  │
│                             X' ─→ FFN(X') ─→ + X' ─→ LayerNorm ─→ 出力
```

1. Layer Normalization（層正規化）
2. Feed-Forward Network（FFN）
3. Residual Connection（残差接続）
4. N 層スタック
5. Language Model Head（出力層）

**使うライブラリ**：numpy のみ

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

# Notebook 01/02 と同じ設定・関数をすべて再定義
vocab_size = 8
d_model    = 4
seq_len    = 4
num_heads  = 2
d_k_head   = d_model // num_heads   # 各ヘッドの次元 = 2

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def positional_encoding(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            angle = pos / (10000 ** (2 * i / d_model))
            PE[pos, i]   = np.sin(angle)
            if i + 1 < d_model:
                PE[pos, i+1] = np.cos(angle)
    return PE

def scaled_dot_product_attention(Q, K, V, mask=True):
    seq_len, d_k = Q.shape
    scores = Q @ K.T / np.sqrt(d_k)
    if mask:
        causal = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
        scores[causal] = -1e9
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

# 入力 X の再現
np.random.seed(42)
E = np.random.randn(vocab_size, d_model)
tokens = [2, 5, 1, 3]
PE = positional_encoding(seq_len, d_model)
X = E[tokens] + PE

# Multi-Head Attention（Notebook 02 で定義済み）
np.random.seed(1)
W_Qs = [np.random.randn(d_model, d_k_head) * 0.5 for _ in range(num_heads)]
W_Ks = [np.random.randn(d_model, d_k_head) * 0.5 for _ in range(num_heads)]
W_Vs = [np.random.randn(d_model, d_k_head) * 0.5 for _ in range(num_heads)]
W_O  = np.random.randn(d_model, d_model) * 0.5

heads = []
for h in range(num_heads):
    Q_h = X @ W_Qs[h]
    K_h = X @ W_Ks[h]
    V_h = X @ W_Vs[h]
    head_out, _ = scaled_dot_product_attention(Q_h, K_h, V_h)
    heads.append(head_out)

mha_output = np.concatenate(heads, axis=-1) @ W_O

print(f"入力 X: shape = {X.shape}")
print(X)
print()
print(f"Multi-Head Attention 出力 mha_output: shape = {mha_output.shape}")
print(mha_output)

入力 X: shape = (4, 4)
[[-0.4695  1.5426 -0.4634  0.5343]
 [ 2.3071  0.3145  0.0676 -0.4247]
 [ 0.6751 -0.6503  1.5794  1.7674]
 [ 0.3831 -2.9033 -1.7246  0.4377]]

Multi-Head Attention 出力 mha_output: shape = (4, 4)
[[-1.2139  1.2118  0.5169 -0.5124]
 [-0.0696 -0.1639 -0.0454 -0.2385]
 [ 0.0691 -0.0957 -0.024  -0.1892]
 [ 0.235   0.0649 -0.0092  0.4202]]


---
## 1. Layer Normalization（層正規化）

各トークンの **特徴次元方向** に正規化します（平均0、分散1）。
学習可能なパラメータ γ（スケール）と β（シフト）も持ちます。

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta$$

- μ, σ²: 各トークンの特徴次元での平均と分散
- ε = 1e-6: ゼロ除算防止
- γ, β: 学習パラメータ（初期値: γ=1, β=0）

**Batch Norm との違い**：Batch Norm はバッチ方向（複数サンプル間）で正規化。
Layer Norm は 1 サンプルの特徴次元方向で正規化（バッチサイズに非依存）。

In [2]:
# Layer Normalization の実装
def layer_norm(x, gamma, beta, eps=1e-6):
    # 各行（各トークン）の特徴次元で平均・分散を計算
    mean = x.mean(axis=-1, keepdims=True)   # (seq_len, 1)
    var  = x.var(axis=-1, keepdims=True)    # (seq_len, 1)
    x_hat = (x - mean) / np.sqrt(var + eps) # 正規化
    return gamma * x_hat + beta             # スケール + シフト

# 初期パラメータ
gamma = np.ones(d_model)
beta  = np.zeros(d_model)

# 手計算での確認（トークン0の行）
row = mha_output[0]
mean_0 = row.mean()
var_0  = row.var()
x_hat_0 = (row - mean_0) / np.sqrt(var_0 + 1e-6)

print("手計算確認: LayerNorm(mha_output[0])")
print(f"  入力      = {row}")
print(f"  平均 μ    = {mean_0:.4f}")
print(f"  分散 σ²   = {var_0:.4f}")
print(f"  (x-μ)/σ   = {x_hat_0}")
print(f"  γ*(x-μ)/σ = {gamma * x_hat_0}")
print()

手計算確認: LayerNorm(mha_output[0])
  入力      = [-1.2139  1.2118  0.5169 -0.5124]
  平均 μ    = 0.0006
  分散 σ²   = 0.8679
  (x-μ)/σ   = [-1.3036  1.3001  0.5542 -0.5506]
  γ*(x-μ)/σ = [-1.3036  1.3001  0.5542 -0.5506]



In [3]:
ln_out = layer_norm(mha_output, gamma, beta)

print(f"LayerNorm(mha_output): shape = {ln_out.shape}")
print(ln_out)
print()
print("各行の平均（≈ 0 になる）:", ln_out.mean(axis=-1))
print("各行の分散（≈ 1 になる）:", ln_out.var(axis=-1))

LayerNorm(mha_output): shape = (4, 4)
[[-1.3036  1.3001  0.5542 -0.5506]
 [ 0.776  -0.4489  1.0899 -1.4169]
 [ 1.3617 -0.377   0.3789 -1.3636]
 [ 0.3455 -0.6811 -1.1284  1.464 ]]

各行の平均（≈ 0 になる）: [ 0.  0. -0. -0.]
各行の分散（≈ 1 になる）: [1.     0.9998 0.9999 1.    ]


---
## 2. Residual Connection + LayerNorm（第1サブレイヤ）

MHA の前後で **残差接続（Add）** を行います：

$$X' = \text{LayerNorm}(X + \text{MHA}(X))$$

残差接続の効果：
- 入力がそのまま出力に流れるため、勾配が深い層まで届きやすい
- 「何もしない（恒等写像）」が学習しやすい

In [4]:
# 第1サブレイヤ: MHA + Add & Norm
X_prime = layer_norm(X + mha_output, gamma, beta)

print("X（入力）:")
print(X)
print()
print("mha_output:")
print(mha_output)
print()
print("X + mha_output（残差接続）:")
print(X + mha_output)
print()
print("X' = LayerNorm(X + mha_output):")
print(X_prime)

X（入力）:
[[-0.4695  1.5426 -0.4634  0.5343]
 [ 2.3071  0.3145  0.0676 -0.4247]
 [ 0.6751 -0.6503  1.5794  1.7674]
 [ 0.3831 -2.9033 -1.7246  0.4377]]

mha_output:
[[-1.2139  1.2118  0.5169 -0.5124]
 [-0.0696 -0.1639 -0.0454 -0.2385]
 [ 0.0691 -0.0957 -0.024  -0.1892]
 [ 0.235   0.0649 -0.0092  0.4202]]

X + mha_output（残差接続）:
[[-1.6833  2.7543  0.0535  0.0219]
 [ 2.2376  0.1506  0.0223 -0.6633]
 [ 0.7442 -0.746   1.5554  1.5783]
 [ 0.6181 -2.8383 -1.7338  0.858 ]]

X' = LayerNorm(X + mha_output):
[[-1.24    1.5534 -0.1467 -0.1666]
 [ 1.6601 -0.2638 -0.3821 -1.0141]
 [-0.041  -1.6188  0.8178  0.842 ]
 [ 0.8901 -1.3199 -0.6137  1.0435]]


---
## 3. Feed-Forward Network（FFN）

各トークンに **同じ** 2 層 MLP を独立に適用します。

$$\text{FFN}(x) = \text{GELU}(x W_1 + b_1) W_2 + b_2$$

- $W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$：次元を拡大（通常 4 倍）
- $W_2 \in \mathbb{R}^{d_{\text{ff}} \times d_{\text{model}}}$：元の次元に戻す
- GELU：`0.5x(1 + tanh(√(2/π)(x + 0.044715x³)))` — ReLU の滑らかな版

**なぜ FFN が必要か**：Attention は「どこを見るか」を学習するが、
FFN は「何を計算するか（特徴変換）」を学習する役割を担う。

In [5]:
d_ff = 8   # d_model の 2 倍（実際は 4 倍が多い）

np.random.seed(2)
W1 = np.random.randn(d_model, d_ff) * 0.3
b1 = np.zeros(d_ff)
W2 = np.random.randn(d_ff, d_model) * 0.3
b2 = np.zeros(d_model)

print(f"W1: shape = {W1.shape}  （{d_model} → {d_ff}）")
print(W1)
print()
print(f"W2: shape = {W2.shape}  （{d_ff} → {d_model}）")

W1: shape = (4, 8)  （4 → 8）
[[-0.125  -0.0169 -0.6409  0.4921 -0.538  -0.2525  0.1509 -0.3736]
 [-0.3174 -0.2727  0.1654  0.6877  0.0125 -0.3354  0.1617 -0.1788]
 [-0.0057  0.3525 -0.2244  0.0027 -0.2634 -0.0469  0.077  -0.2966]
 [-0.1016 -0.0709 -0.1913 -0.3563 -0.4264 -0.046  -0.0807  0.6694]]

W2: shape = (8, 4)  （8 → 4）


In [6]:
def gelu(x):
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))

def ffn(x, W1, b1, W2, b2):
    h = gelu(x @ W1 + b1)   # (seq_len, d_ff)
    return h @ W2 + b2      # (seq_len, d_model)

# 手計算：トークン0 の中間値を確認
row = X_prime[0]
h0  = gelu(row @ W1 + b1)   # 中間層の活性化

print("手計算確認: FFN(X'[0])")
print(f"  X'[0]       = {row}")
print(f"  X'[0] @ W1  = {row @ W1}")
print(f"  + b1        = {row @ W1 + b1}")
print(f"  GELU(...)   = {h0}  ← 中間層出力")
print(f"  h0 @ W2 + b2= {h0 @ W2 + b2}")
print()

ffn_output = ffn(X_prime, W1, b1, W2, b2)
print(f"FFN 出力 ffn_output: shape = {ffn_output.shape}")
print(ffn_output)

手計算確認: FFN(X'[0])
  X'[0]       = [-1.24    1.5534 -0.1467 -0.1666]
  X'[0] @ W1  = [-0.3202 -0.4426  1.1165  0.517   0.7962 -0.1933  0.0663  0.1174]
  + b1        = [-0.3202 -0.4426  1.1165  0.517   0.7962 -0.1933  0.0663  0.1174]
  GELU(...)   = [-0.1199 -0.1456  0.9688  0.3605  0.6266 -0.0818  0.0349  0.0642]  ← 中間層出力
  h0 @ W2 + b2= [-0.0281  0.2417 -0.3112  0.3195]

FFN 出力 ffn_output: shape = (4, 4)
[[-0.0281  0.2417 -0.3112  0.3195]
 [ 0.4222 -0.0926  0.2062 -0.208 ]
 [-0.1993 -0.1905  0.0845  0.3596]
 [-0.1724 -0.0699  0.0276  0.1361]]


In [7]:
# GELU vs ReLU の比較
x_vals = np.linspace(-3, 3, 100)
gelu_vals = gelu(x_vals)
relu_vals = np.maximum(0, x_vals)

print("GELU(-2.0) =", gelu(-2.0))
print("GELU(-0.5) =", gelu(-0.5))
print("GELU( 0.0) =", gelu(0.0))
print("GELU( 1.0) =", gelu(1.0))
print("GELU( 2.0) =", gelu(2.0))
print()
print("ReLU との比較: GELU は x<0 でも微小な値を通す（滑らか）")

GELU(-2.0) = -0.04540230591222494
GELU(-0.5) = -0.15428599017485606
GELU( 0.0) = 0.0
GELU( 1.0) = 0.8411919906082768
GELU( 2.0) = 1.954597694087775

ReLU との比較: GELU は x<0 でも微小な値を通す（滑らか）


---
## 4. 第 2 サブレイヤ: FFN + Add & Norm

$$\text{出力} = \text{LayerNorm}(X' + \text{FFN}(X'))$$

In [8]:
gamma2 = np.ones(d_model)
beta2  = np.zeros(d_model)

# 第2サブレイヤ: FFN + Add & Norm
X_out = layer_norm(X_prime + ffn_output, gamma2, beta2)

print("X'（MHA後）:")
print(X_prime)
print()
print("ffn_output:")
print(ffn_output)
print()
print("Transformer ブロック出力 X_out = LayerNorm(X' + FFN(X'))")
print(f"shape = {X_out.shape}")
print(X_out)
print()
print("入力 X と出力 X_out は同じ形状 — 層を何層重ねても shape が変わらない")

X'（MHA後）:
[[-1.24    1.5534 -0.1467 -0.1666]
 [ 1.6601 -0.2638 -0.3821 -1.0141]
 [-0.041  -1.6188  0.8178  0.842 ]
 [ 0.8901 -1.3199 -0.6137  1.0435]]

ffn_output:
[[-0.0281  0.2417 -0.3112  0.3195]
 [ 0.4222 -0.0926  0.2062 -0.208 ]
 [-0.1993 -0.1905  0.0845  0.3596]
 [-0.1724 -0.0699  0.0276  0.1361]]

Transformer ブロック出力 X_out = LayerNorm(X' + FFN(X'))
shape = (4, 4)
[[-1.1778  1.548  -0.4569  0.0867]
 [ 1.6387 -0.3591 -0.2112 -1.0683]
 [-0.2148 -1.5422  0.7519  1.0051]
 [ 0.7213 -1.3403 -0.5541  1.1731]]

入力 X と出力 X_out は同じ形状 — 層を何層重ねても shape が変わらない


---
## 5. Transformer ブロックのクラス化

ここまでの処理をクラスにまとめます。

In [9]:
class MultiHeadAttention:
    def __init__(self, d_model, num_heads, seed=0):
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        rng = np.random.default_rng(seed)
        self.W_Qs = [rng.standard_normal((d_model, self.d_k)) * 0.5 for _ in range(num_heads)]
        self.W_Ks = [rng.standard_normal((d_model, self.d_k)) * 0.5 for _ in range(num_heads)]
        self.W_Vs = [rng.standard_normal((d_model, self.d_k)) * 0.5 for _ in range(num_heads)]
        self.W_O  = rng.standard_normal((d_model, d_model)) * 0.5

    def forward(self, X):
        heads = []
        for h in range(self.num_heads):
            Q_h = X @ self.W_Qs[h]
            K_h = X @ self.W_Ks[h]
            V_h = X @ self.W_Vs[h]
            head_out, _ = scaled_dot_product_attention(Q_h, K_h, V_h)
            heads.append(head_out)
        return np.concatenate(heads, axis=-1) @ self.W_O


class LayerNorm:
    def __init__(self, d_model, eps=1e-6):
        self.gamma = np.ones(d_model)
        self.beta  = np.zeros(d_model)
        self.eps   = eps

    def forward(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        var  = x.var(axis=-1, keepdims=True)
        return self.gamma * (x - mean) / np.sqrt(var + self.eps) + self.beta


class FeedForward:
    def __init__(self, d_model, d_ff, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.standard_normal((d_model, d_ff)) * 0.3
        self.b1 = np.zeros(d_ff)
        self.W2 = rng.standard_normal((d_ff, d_model)) * 0.3
        self.b2 = np.zeros(d_model)

    def forward(self, x):
        h = gelu(x @ self.W1 + self.b1)
        return h @ self.W2 + self.b2


class TransformerBlock:
    def __init__(self, d_model, num_heads, d_ff, seed=0):
        self.mha  = MultiHeadAttention(d_model, num_heads, seed=seed)
        self.ffn  = FeedForward(d_model, d_ff, seed=seed+1)
        self.ln1  = LayerNorm(d_model)
        self.ln2  = LayerNorm(d_model)

    def forward(self, X):
        # 第1サブレイヤ: MHA + Add & Norm
        X = self.ln1.forward(X + self.mha.forward(X))
        # 第2サブレイヤ: FFN + Add & Norm
        X = self.ln2.forward(X + self.ffn.forward(X))
        return X

block = TransformerBlock(d_model=4, num_heads=2, d_ff=8, seed=42)
output = block.forward(X)
print(f"TransformerBlock 入力: shape = {X.shape}")
print(f"TransformerBlock 出力: shape = {output.shape}")
print(output)

TransformerBlock 入力: shape = (4, 4)
TransformerBlock 出力: shape = (4, 4)
[[-1.4808  1.2866 -0.1606  0.3549]
 [ 1.3931  0.4105 -0.5383 -1.2653]
 [-0.1428 -1.487   0.3456  1.2841]
 [ 0.9781 -1.153  -0.8341  1.009 ]]


---
## 6. N 層スタック

Transformer は同じブロックを N 回重ねます。
各ブロックの重みは独立で、前のブロックの出力が次のブロックの入力になります。

In [10]:
n_layers = 3

blocks = [TransformerBlock(d_model=4, num_heads=2, d_ff=8, seed=layer*10)
          for layer in range(n_layers)]

current = X.copy()
print(f"入力 X: shape = {X.shape}")
print()

for i, block in enumerate(blocks):
    current = block.forward(current)
    print(f"Layer {i+1} 後: shape = {current.shape}")
    print(current)
    print()

print(f"最終出力（{n_layers} 層）: shape = {current.shape}")
print("→ 入力と同じ shape を維持しながら層を重ねられる")

入力 X: shape = (4, 4)

Layer 1 後: shape = (4, 4)
[[-0.2682  1.6892 -0.8906 -0.5305]
 [ 1.4698  0.1243 -0.2709 -1.3232]
 [ 0.7808 -1.7176  0.4934  0.4434]
 [ 0.8233 -1.6307 -0.0068  0.8143]]

Layer 2 後: shape = (4, 4)
[[-0.7204  1.201   0.7402 -1.2209]
 [ 1.0392  0.1788  0.4269 -1.6449]
 [ 1.1509 -1.589   0.0536  0.3845]
 [ 1.2769 -1.1866 -0.737   0.6468]]

Layer 3 後: shape = (4, 4)
[[-1.2351  1.3308  0.5433 -0.6389]
 [-0.2956  1.0005  0.8015 -1.5064]
 [ 0.4296 -1.6952  0.3672  0.8983]
 [ 0.7335 -1.2566 -0.6728  1.1959]]

最終出力（3 層）: shape = (4, 4)
→ 入力と同じ shape を維持しながら層を重ねられる


---
## 7. Language Model Head（LM ヘッド）

最終層の出力を **語彙全体のスコア（logits）** に変換します：

$$\text{logits} = X_{\text{out}} W_{\text{lm}} \in \mathbb{R}^{\text{seq\_len} \times \text{vocab\_size}}$$

- `logits[i, j]` = 位置 i の次のトークンとして語彙 j がどれくらい適切か
- これに softmax を適用すると「次トークンの確率分布」になる

In [11]:
np.random.seed(99)
W_lm = np.random.randn(d_model, vocab_size) * 0.3

logits = current @ W_lm   # (seq_len, vocab_size) = (4, 8)

print(f"W_lm: shape = {W_lm.shape}  （{d_model} → {vocab_size}）")
print()
print(f"logits: shape = {logits.shape}")
print(logits)
print()

# 各位置の「次トークン確率分布」
probs = softmax(logits, axis=-1)   # (4, 8)
print(f"確率 probs: shape = {probs.shape}")
print(probs)
print()
print("各行の合計:", probs.sum(axis=-1))   # すべて 1.0

W_lm: shape = (4, 8)  （4 → 8）

logits: shape = (4, 8)
[[ 0.2231 -1.7136 -0.0969 -0.1141 -0.0758  0.0729 -0.2013 -0.9195]
 [ 0.3597 -0.8238  0.1731  0.2503 -0.21   -0.2125  0.0678 -0.5728]
 [ 0.0357  1.2809 -0.1538 -0.1242 -0.0798  0.0375 -0.2891  0.8297]
 [-0.3009  1.2967 -0.0664 -0.1012  0.1483  0.0922  0.0311  0.7794]]

確率 probs: shape = (4, 8)
[[0.1938 0.0279 0.1407 0.1383 0.1437 0.1668 0.1268 0.0618]
 [0.1886 0.0577 0.1565 0.169  0.1067 0.1064 0.1408 0.0742]
 [0.0911 0.3163 0.0753 0.0776 0.0811 0.0912 0.0658 0.2015]
 [0.0632 0.3124 0.0799 0.0772 0.0991 0.0937 0.0881 0.1863]]

各行の合計: [1. 1. 1. 1.]


In [12]:
# 各位置での予測（argmax = 最も確率の高い次トークン）
predicted_ids = probs.argmax(axis=-1)
print("Greedy 予測（各位置の次トークン）:")
for i, (inp, pred) in enumerate(zip(tokens, predicted_ids)):
    print(f"  位置 {i} (token_id={inp}) → 予測次トークン ID = {pred}  "
          f"(確率 {probs[i, pred]:.4f})")

Greedy 予測（各位置の次トークン）:
  位置 0 (token_id=2) → 予測次トークン ID = 0  (確率 0.1938)
  位置 1 (token_id=5) → 予測次トークン ID = 0  (確率 0.1886)
  位置 2 (token_id=1) → 予測次トークン ID = 1  (確率 0.3163)
  位置 3 (token_id=3) → 予測次トークン ID = 1  (確率 0.3124)


---
## 8. パラメータ数の計算

モデルの規模感を把握するためにパラメータ数を数えます。

In [13]:
def count_params_block(block):
    mha = block.mha
    ffn = block.ffn
    mha_params = sum(w.size for w in mha.W_Qs + mha.W_Ks + mha.W_Vs) + mha.W_O.size
    ffn_params  = ffn.W1.size + ffn.b1.size + ffn.W2.size + ffn.b2.size
    ln_params   = (block.ln1.gamma.size + block.ln1.beta.size +
                   block.ln2.gamma.size + block.ln2.beta.size)
    return mha_params, ffn_params, ln_params

mha_p, ffn_p, ln_p = count_params_block(blocks[0])
print(f"1 ブロックのパラメータ数:")
print(f"  Multi-Head Attention: {mha_p}")
print(f"  Feed-Forward Network: {ffn_p}")
print(f"  Layer Normalization:  {ln_p}")
print(f"  合計 (1ブロック):     {mha_p + ffn_p + ln_p}")
print()
print(f"全 {n_layers} 層:      {(mha_p + ffn_p + ln_p) * n_layers}")
print(f"Embedding テーブル:    {vocab_size * d_model}")
print(f"LM Head:               {d_model * vocab_size}")
print(f"総パラメータ数:         {(mha_p + ffn_p + ln_p) * n_layers + vocab_size * d_model + d_model * vocab_size}")
print()
print("参考: GPT-2（small）≈ 117,000,000、GPT-4 は公開非公開ながら推定数百B")

1 ブロックのパラメータ数:
  Multi-Head Attention: 64
  Feed-Forward Network: 76
  Layer Normalization:  16
  合計 (1ブロック):     156

全 3 層:      468
Embedding テーブル:    32
LM Head:               32
総パラメータ数:         532

参考: GPT-2（small）≈ 117,000,000、GPT-4 は公開非公開ながら推定数百B


---
## まとめ

Transformer ブロック全体の計算フロー：

```
X (4, 4)
 │
 ├─→ MultiHeadAttention(X) ──→ X + … ─→ LayerNorm ─→ X' (4, 4)
 │                                                     │
 │              X' ─→ FFN(X') ──→ X' + … ─→ LayerNorm ─→ X_out (4, 4)
 │
 └── （これを N 層繰り返す）
                                                     │
                                              LM Head (@ W_lm)
                                                     │
                                           logits (4, vocab_size)
```

→ **次のノートブック `04_training_inference.ipynb`** では、
この logits から損失を計算し、勾配降下でパラメータを更新し、テキスト生成を行います。